#  BEREINIGUNG – items.csv

Ziel: Den Produktkatalog (items) in einen sauberen,
analysierbaren Zustand bringen, bevor er mit dem
Transaktionsdatensatz (train) zusammengeführt wird.

Der items-Datensatz enthält 22'035 Produkte mit 11 Spalten:
pid, manufacturer, group, content, unit, pharmForm,
genericProduct, salesIndex, category, campaignIndex, rrp

In [44]:
## 1. Einlesen der Stammdaten (items.csv)
import pandas as pd

items = pd.read_csv('C:/Users/anith/PycharmProjects/DataPreProcessing_final/data/items.csv', header=0, sep='|')

items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22035 entries, 0 to 22034
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pid             22035 non-null  int64  
 1   manufacturer    22035 non-null  int64  
 2   group           22035 non-null  object 
 3   content         22035 non-null  object 
 4   unit            22035 non-null  object 
 5   pharmForm       19708 non-null  object 
 6   genericProduct  22035 non-null  int64  
 7   salesIndex      22035 non-null  int64  
 8   category        17408 non-null  float64
 9   campaignIndex   1338 non-null   object 
 10  rrp             22035 non-null  float64
dtypes: float64(2), int64(4), object(5)
memory usage: 1.8+ MB


## 1.1 – DUPLIKATE
Jede Zeile in items repräsentiert ein eindeutiges Produkt
(pid = Produkt-ID). Doppelte Einträge würden beim späteren
Merge zu künstlich verdoppelten Transaktionen führen und
damit alle Aggregationen verfälschen.
→ Duplikate werden geprüft und entfernt.

In [45]:
#1.1 – DUPLIKATE
print(f"Duplikate in items: {items.duplicated().sum()}")

Duplikate in items: 0


## 1.2 – MISSING VALUES

In [46]:
missing = items.isnull().sum()
missing_pct = (missing / len(items) * 100).round(2)
pd.DataFrame({'Anzahl': missing, 'Prozent': missing_pct})

,Anzahl,Prozent
pid,0,0.00
manufacturer,0,0.00
group,0,0.00
content,0,0.00
unit,0,0.00
pharmForm,2327,10.56
genericProduct,0,0.00
salesIndex,0,0.00
category,4627,21.00
campaignIndex,20697,93.93



Drei Spalten weisen fehlende Werte auf:

**pharmForm**     (2'327 fehlend, ~11%):
Die pharmazeutische Form ist nicht für alle Produkte
angegeben. Da es sich um eine kategoriale Variable handelt,
werden fehlende Werte mit "MISSING" aufgefüllt, um die
Information "kein Eintrag vorhanden" explizit zu machen.

**category**      (4'627 fehlend, ~21%):
Die Produktkategorie fehlt bei einem Fünftel der Produkte.
Fehlende Werte erhalten den Wert -1 als eigene Kategorie,
da das Fehlen selbst eine inhaltliche Aussage sein kann
(z.B. Produkte ohne Kategorisierung).
Zusätzlich wird ein binäres Flag category_missing erstellt.

**campaignIndex** (20'697 fehlend, ~94%):
Der campaignIndex ist nur für wenige Aktionsprodukte
vergeben. Das Fehlen bedeutet schlicht: kein Werbeeinsatz.
→ Ein binäres Flag has_campaign (1/0) wird erstellt.
→ Fehlende Werte werden mit "NONE" aufgefüllt.

In [47]:
#pharmForm
items['pharmForm_norm'] = items['pharmForm'].fillna('MISSING')
items['pharmForm_norm'] = items['pharmForm_norm'].str.upper().str.strip()
print(f"Vorher – fehlende Werte: {items['pharmForm'].isnull().sum()}")
print(f"Nachher – fehlende Werte: {items['pharmForm_norm'].isnull().sum()}")
print(f"\nVorher:\n{items['pharmForm'].value_counts()}")
print(f"\nNachher:\n{items['pharmForm_norm'].value_counts()}")

Vorher – fehlende Werte: 2327
Nachher – fehlende Werte: 0

Vorher:
pharmForm
TAB    1944
CRE    1538
KAP    1460
GLO     951
TRO     905
       ... 
Vka       1
Inl       1
Ple       1
Fse       1
HVW       1
Name: count, Length: 278, dtype: int64

Nachher:
pharmForm_norm
MISSING    2327
TAB        1993
CRE        1576
KAP        1503
GLO         971
           ... 
IFL           1
GPA           1
APA           1
AUC           1
HVW           1
Name: count, Length: 184, dtype: int64


Umwandlung category: float64 → int → category

Die Spalte category ist ursprünglich als float64 gespeichert
(z.B. 3.0, 7.0), obwohl sie inhaltlich eine Produktgruppe
repräsentiert. Eine Dezimalzahl hat hier keine Bedeutung.

Der Zwischenschritt float → int bereinigt die Darstellung:
aus 3.0 wird 3, aus -1.0 wird -1 (fehlende Werte).

Der finale Schritt int → category signalisiert explizit,
dass es sich um eine Gruppe handelt und keine Zahl mit der
gerechnet werden kann. Zudem reduziert der category-Dtype
den Speicherbedarf erheblich, da pandas jeden Wert nur
einmal speichert und intern nur noch einen Index verwaltet.

In [48]:
#category
items['category_norm'] = items['category'].fillna(-1).astype(int).astype('category')

# Kontrolle
print(f"Vorher  – Dtype: {items['category'].dtype}, fehlende Werte: {items['category'].isnull().sum()}")
print(f"Nachher – Dtype: {items['category_norm'].dtype}, fehlende Werte: {items['category_norm'].isnull().sum()}")
print(f"\nKategorien: {items['category_norm'].nunique()} (inkl. -1 für fehlend)")
print(items['category_norm'].value_counts().head(10))

Vorher  – Dtype: float64, fehlende Werte: 4627
Nachher – Dtype: category, fehlende Werte: 0

Kategorien: 410 (inkl. -1 für fehlend)
category_norm
-1     4627
 3      385
 12     316
 76     312
 80     290
 23     288
 8      278
 24     260
 25     230
 73     212
Name: count, dtype: int64


Zusammenfassen seltener Kategorien → OTHER

Kategorien mit weniger als 10 Einträgen werden zur Gruppe
OTHER zusammengefasst. Von 410 Kategorien sind 132 betroffen.

Begründung:
Seltene Kategorien kommen in items weniger als 10 Mal vor.
Da train 2.5 Mio. Transaktionen enthält, sieht das Modell
diese Kategorien extrem selten und kann daraus kein
verlässliches Muster lernen. Dies führt zu zwei Problemen:

Overfitting: Das Modell memoriert seltene Kategorien anstatt
zu generalisieren, was die Vorhersagequalität auf neuen Daten
verschlechtert.

Unseen Categories: Taucht eine seltene Kategorie im Test-Set
auf, kennt das Modell sie nicht und liefert fehlerhafte
Vorhersagen.

Durch OTHER werden diese Produkte explizit als Gruppe mit
unzureichender Datenbasis markiert und gemeinsam behandelt.

In [49]:
#seltene Kategorien zusammenfassen zu "Other"
freq = items['category_norm'].value_counts()
print(f"Kategorien mit < 5 Einträgen:  {(freq < 5).sum()}")
print(f"Kategorien mit < 10 Einträgen: {(freq < 10).sum()}")
print(f"Kategorien mit < 20 Einträgen: {(freq < 20).sum()}")

Kategorien mit < 5 Einträgen:  75
Kategorien mit < 10 Einträgen: 132
Kategorien mit < 20 Einträgen: 194


In [50]:
SCHWELLWERT = 10

freq = items['category_norm'].value_counts()
rare = freq[freq < SCHWELLWERT].index

items['category_norm'] = items['category_norm'].cat.add_categories('OTHER')
items.loc[items['category_norm'].isin(rare), 'category_norm'] = 'OTHER'
items['category_norm'] = items['category_norm'].cat.remove_unused_categories()

print(f"Kategorien vorher:  410")
print(f"Kategorien nachher: {items['category_norm'].nunique()}")

Kategorien vorher:  410
Kategorien nachher: 279


Behandlung campaignIndex

campaignIndex ist nur für Aktionsprodukte vergeben. Das Fehlen bedeutet
inhaltlich: kein Werbeeinsatz für dieses Produkt.

has_campaign (0/1): Binäres Flag das explizit festhält
ob ein Produkt beworben wird. Da Kampagnen den Kaufentscheid
beeinflussen können, ist diese Information als eigenes
Feature wertvoll.

campaignIndex_norm: Fehlende Werte werden mit NONE
aufgefüllt und die Spalte zu category umgewandelt.
NONE signalisiert explizit "kein Werbeeinsatz" und
wird vom Modell als eigene Gruppe behandelt.

In [51]:
#campaignIndex
# Flag erstellen: 1 = hat Kampagne, 0 = keine Kampagne
items['has_campaign'] = items['campaignIndex'].notna().astype(int)

# Fehlende Werte mit NONE auffüllen + zu category umwandeln
items['campaignIndex_norm'] = items['campaignIndex'].fillna('NONE').astype('category')

# Kontrolle
print(f"has_campaign Verteilung:\n{items['has_campaign'].value_counts()}")
print(f"\nVorher  – fehlende Werte: {items['campaignIndex'].isnull().sum()}")
print(f"Nachher – fehlende Werte: {items['campaignIndex_norm'].isnull().sum()}")
print(f"\nKategorien: {items['campaignIndex_norm'].value_counts()}")

has_campaign Verteilung:
has_campaign
0    20697
1     1338
Name: count, dtype: int64

Vorher  – fehlende Werte: 20697
Nachher – fehlende Werte: 0

Kategorien: campaignIndex_norm
NONE    20697
B         738
A         350
C         250
Name: count, dtype: int64


## 1.3 zusätzliche Vorbereitungen: content, unit, salesindex

In [52]:
import numpy as np
# is_multipack
items['is_multipack'] = (
    items['content'].str.contains(r'\d+[Xx]\d+', na=False)
).astype(int)

# pack_n: erste × zweite Zahl bei 3-teilig, erste Zahl bei 2-teilig, sonst 1
items['pack_n'] = items['content'].str.extract(r'^(\d+)[Xx](\d+)[Xx]\d+').apply(
    lambda x: int(x[0]) * int(x[1]) if pd.notna(x[0]) else np.nan, axis=1
).fillna(
    items['content'].str.extract(r'^(\d+)[Xx]\d+$')[0].astype(float)
).fillna(1).astype(int)

# pack_size: immer die letzte Zahl
items['pack_size'] = items['content'].str.extract(r'(\d+\.?\d*)$')[0].astype(float)

# Kontrolle
print(items[['content', 'unit', 'is_multipack', 'pack_n', 'pack_size']].head(20))
print(f"\nMultipacks:         {items['is_multipack'].sum()}")
print(f"Fehlende pack_size: {items['pack_size'].isna().sum()}")




   content unit  is_multipack  pack_n  pack_size
0       80   ST             0       1       80.0
1       80   ST             0       1       80.0
2       10    G             0       1       10.0
3       80   ST             0       1       80.0
4        8   ST             0       1        8.0
5       10    G             0       1       10.0
6       15   ST             0       1       15.0
7       80   ST             0       1       80.0
8       10    G             0       1       10.0
9       30   ST             0       1       30.0
10      60   ST             0       1       60.0
11      14   ST             0       1       14.0
12      30   ML             0       1       30.0
13       7   ST             0       1        7.0
14      14   ST             0       1       14.0
15       8   ST             0       1        8.0
16      75   ML             0       1       75.0
17      30   ML             0       1       30.0
18      10   ML             0       1       10.0
19      10   ML     

In [53]:
plausibility = {
    'ML': (0.1, 5000),
    'G':  (0.1, 5000),
    'CM': (0.1, 500),
    'ST': (1,   5000),
    'TL': (1,   500),
}

mask = items.apply(
    lambda row: pd.notna(row['pack_size'])
    and row['unit'].upper() in plausibility
    and not (plausibility[row['unit'].upper()][0]
             <= row['pack_size']
             <= plausibility[row['unit'].upper()][1]),
    axis=1
)

print(f"Unplausible Zeilen: {mask.sum()}")
print(items[mask][['content', 'unit', 'pack_size']])

Unplausible Zeilen: 0
Empty DataFrame
Columns: [content, unit, pack_size]
Index: []


In [54]:
# unit harmonisieren: l → ml, kg → g, m → cm
unit_mapping = {
    'l':  'ml',
    'kg': 'g',
    'm':  'cm'
}
conversion_factor = {
    'l':  1000,
    'kg': 1000,
    'm':  100
}

# Einheit normalisieren
items['unit_norm'] = items['unit'].str.lower().replace(unit_mapping)

# pack_size anpassen falls Einheit konvertiert wurde
items['pack_size'] = items.apply(
    lambda row: row['pack_size'] * conversion_factor[row['unit'].lower()]
    if row['unit'].lower() in conversion_factor else row['pack_size'],
    axis=1
)

# Kontrolle
print(items[items['unit'].str.lower().isin(unit_mapping)][
    ['content', 'unit', 'unit_norm', 'pack_size']
].head(10))
print(f"\nunit_norm Verteilung:\n{items['unit_norm'].value_counts()}")


     content unit unit_norm  pack_size
1029       1   KG         g     1000.0
2125     1.5   KG         g     1500.0
2855      30    M        cm     3000.0
3164       1    L        ml     1000.0
3167     1X2    L        ml     2000.0
3620       1    L        ml     1000.0
3658       1    L        ml     1000.0
3659       1    L        ml     1000.0
3691       1    L        ml     1000.0
3900      10    L        ml    10000.0

unit_norm Verteilung:
unit_norm
st    10061
ml     7942
g      3884
p       134
cm       14
Name: count, dtype: int64


In [55]:
items['salesIndex_norm'] = items['salesIndex'].astype('category')

# Kontrolle
print(f"Dtype vorher:  {items['salesIndex'].dtype}")
print(f"Dtype nachher: {items['salesIndex_norm'].dtype}")
print(items['salesIndex_norm'].value_counts().sort_index())

Dtype vorher:  int64
Dtype nachher: category
salesIndex_norm
40     7690
44      125
52      983
53    13237
Name: count, dtype: int64


In [56]:
# Längenverteilung nochmals anschauen
print(items['group'].str.len().value_counts().sort_index())

# Erste 2 Zeichen
print(f"\nErste 2 Zeichen – unique: {items['group'].str[:2].nunique()}")
print(items['group'].str[:2].value_counts().head(10))

# Erste 3 Zeichen
print(f"\nErste 3 Zeichen – unique: {items['group'].str[:3].nunique()}")
print(items['group'].str[:3].value_counts().head(10))

# Erste 4 Zeichen
print(f"\nErste 4 Zeichen – unique: {items['group'].str[:4].nunique()}")
print(items['group'].str[:4].value_counts().head(10))

group
2      305
4     4089
5    11982
6     1438
8     4221
Name: count, dtype: int64

Erste 2 Zeichen – unique: 21
group
22    5491
21    4190
2F    3590
20    2269
10    1697
13     940
1C     857
19     522
18     439
1D     358
Name: count, dtype: int64

Erste 3 Zeichen – unique: 26
group
22O    5491
21O    4190
2FO    3520
20O    2269
10O    1085
1CO     857
13O     813
10I     612
19O     522
18O     439
Name: count, dtype: int64

Erste 4 Zeichen – unique: 87
group
22OI    4298
2FOI    3099
21OK    2637
20OH    1228
20OI     723
22OZ     718
21OI     653
2FOZ     421
1COS     414
21OS     402
Name: count, dtype: int64


Behandlung group

Die Spalte group enthält alphanumerische Produktgruppen-Codes
unterschiedlicher Länge (z.B. 2FOI, 22OI3, 1DOIF0ZO).
Mit 533 unique Werten ist die Kardinalität zu hoch für ein
direktes Encoding.

Analyse der Hierarchiestruktur:
Eine Untersuchung der Zeichenlängen zeigt eine klare
hierarchische Struktur:

   1 Zeichen  →   5 Gruppen  (grobe Hauptkategorie)
   2 Zeichen  →  21 Gruppen  (Untergruppe)
   3 Zeichen  →  26 Gruppen  (kaum Mehrwert, 3. Zeichen meist "O")
   4 Zeichen  →  87 Gruppen  (zu fein, erhöht Komplexität)

Entscheidung: Erste 2 Zeichen als group_norm
L3 (3 Zeichen) bringt gegenüber L2 kaum Mehrwert, da das
dritte Zeichen fast durchgehend ein konstantes "O" ist und
keine inhaltliche Information trägt.
L4 (4 Zeichen) mit 87 Gruppen wäre zu granular.

Die ersten 2 Zeichen mit 21 Gruppen bieten den besten
Kompromiss zwischen Informationsgehalt und Modellkomplexität.

In [57]:
items['group_norm'] = items['group'].str[:2].astype('category')

# Kontrolle
print(f"Kategorien: {items['group_norm'].nunique()}")
print(items['group_norm'].value_counts())

Kategorien: 21
group_norm
22    5491
21    4190
2F    3590
20    2269
10    1697
13     940
1C     857
19     522
18     439
1D     358
14     326
1E     314
12     279
11     228
23     191
24      98
2E      82
2G      74
1A      36
17      28
15      26
Name: count, dtype: int64


### 1.3 – HOHE KARDINALITÄT (manufacturer)

Kardinalität bezeichnet die Anzahl einzigartiger Werte
einer Spalte. Hohe Kardinalität entsteht wenn eine Spalte
sehr viele verschiedene Werte enthält, z.B. hunderte von
Hersteller-IDs oder Produkt-IDs.

Problem:
Spalten mit hoher Kardinalität führen zu zwei Problemen:

Overfitting: Das Modell lernt seltene Werte auswendig
anstatt allgemeine Muster zu erkennen.

Speicher & Komplexität: Viele Kategorien erhöhen die
Modellkomplexität unnötig, besonders beim Encoding.

Vorgehen:
1. Überblick: Kardinalität aller Spalten prüfen
2. Betroffene Spalten identifizieren (> 50 unique Werte)
3. Seltene Werte (< 10 Einträge) zu OTHER zusammenfassen

In items ist manufacturer mit 532 unique IDs betroffen.
pid wird bewusst nicht behandelt da sie nur als
Merge-Schlüssel dient und kein Modell-Feature ist.

In [58]:
# Überblick über alle Spalten
kardinalitaet = pd.DataFrame({
    'unique_werte': items.nunique(),
    'dtype': items.dtypes,
    'pct_unique': (items.nunique() / len(items) * 100).round(2)
}).sort_values('unique_werte', ascending=False)

print(kardinalitaet)


                    unique_werte     dtype  pct_unique
pid                        22035     int64      100.00
rrp                         3289   float64       14.93
manufacturer                1067     int64        4.84
content                      548    object        2.49
group                        533    object        2.42
category                     409   float64        1.86
category_norm                279  category        1.27
pharmForm                    278    object        1.26
pack_size                    227   float64        1.03
pharmForm_norm               184    object        0.84
pack_n                        29     int64        0.13
group_norm                    21  category        0.10
unit                           8    object        0.04
unit_norm                      5    object        0.02
campaignIndex_norm             4  category        0.02
salesIndex                     4     int64        0.02
salesIndex_norm                4  category        0.02
campaignIn

In [59]:
freq = items['manufacturer'].value_counts()
print(freq.describe())
print(f"\nHersteller mit < 5 Einträgen:  {(freq < 5).sum()}")
print(f"Hersteller mit < 10 Einträgen: {(freq < 10).sum()}")
print(f"Hersteller mit < 20 Einträgen: {(freq < 20).sum()}")
print(f"\nTop 10 Hersteller:\n{freq.head(10)}")

count    1067.000000
mean       20.651359
std        55.515817
min         1.000000
25%         2.000000
50%         4.000000
75%        18.000000
max      1258.000000
Name: count, dtype: float64

Hersteller mit < 5 Einträgen:  539
Hersteller mit < 10 Einträgen: 692
Hersteller mit < 20 Einträgen: 820

Top 10 Hersteller:
manufacturer
1      1258
92      324
60      315
41      310
37      303
18      285
176     283
90      266
193     246
102     243
Name: count, dtype: int64


In [60]:
# Anteil Generika pro Hersteller
generic_check = items.groupby('manufacturer').agg(
    anzahl_produkte = ('pid', 'count'),
    anteil_generika = ('genericProduct', 'mean')
).reset_index()

# Nur seltene Hersteller anschauen
rare = freq[freq < 20].index
generic_rare = generic_check[generic_check['manufacturer'].isin(rare)]

print(f"Seltene Hersteller total: {len(generic_rare)}")
print(f"\nAnteil Generika bei seltenen Herstellern:")
print(generic_rare['anteil_generika'].describe())
print(f"\nNur Generika-Hersteller (100% Generika): {(generic_rare['anteil_generika'] == 1).sum()}")
print(f"Keine Generika (0% Generika):            {(generic_rare['anteil_generika'] == 0).sum()}")
print(f"Gemischt:                                {((generic_rare['anteil_generika'] > 0) & (generic_rare['anteil_generika'] < 1)).sum()}")

Seltene Hersteller total: 820

Anteil Generika bei seltenen Herstellern:
count    820.000000
mean       0.017064
std        0.105515
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: anteil_generika, dtype: float64

Nur Generika-Hersteller (100% Generika): 6
Keine Generika (0% Generika):            785
Gemischt:                                29


Behandlung manufacturer

Die Spalte manufacturer enthält anonymisierte Hersteller-IDs
mit 1067 unique Werten und einem ausgeprägten Long Tail:
   - Median: 4 Produkte pro Hersteller
   - 820 von 1067 Herstellern haben weniger als 20 Produkte
   - Max: 1258 Produkte (Hersteller ID 1)

Hersteller mit wenigen Produkten bieten dem Modell kaum
Lernpotenzial und führen zu Overfitting sowie Problemen
mit unbekannten Kategorien im Test-Set.

Vorabprüfung: Generika-Anteil bei seltenen Herstellern
Bevor seltene Hersteller zu OTHER zusammengefasst werden,
wurde geprüft ob diese primär Generika anbieten und
daher eine eigene Kategorie GENERIC rechtfertigen würden.

Ergebnis:
   - 785 von 820 seltenen Herstellern: keine Generika (0%)
   - 6 von 820: ausschliesslich Generika (100%)
   - 29 von 820: gemischt
   - Median Generika-Anteil: 0%

 Eine separate GENERIC Kategorie lohnt sich nicht, da der
 Generika-Anteil bei seltenen Herstellern vernachlässigbar ist.

 Entscheidung: Schwellwert < 20 Produkte → OTHER
 Hersteller mit weniger als 20 Produkten werden zu OTHER
 zusammengefasst. Damit verbleiben 247 Hersteller mit
 ausreichend Daten für das Modell.

In [61]:
freq = items['manufacturer'].value_counts()
rare = freq[freq < 20].index

items['manufacturer_norm'] = items['manufacturer'].astype(str)
items.loc[items['manufacturer'].isin(rare), 'manufacturer_norm'] = 'OTHER'
items['manufacturer_norm'] = items['manufacturer_norm'].astype('category')

print(f"Kategorien vorher:  {items['manufacturer'].nunique()}")
print(f"Kategorien nachher: {items['manufacturer_norm'].nunique()}")
print(f"OTHER-Einträge:     {(items['manufacturer_norm'] == 'OTHER').sum()}")

Kategorien vorher:  1067
Kategorien nachher: 248
OTHER-Einträge:     3825


# 2. BEREINIGUNG – train.csv

Ziel: Den Transaktionsdatensatz (train) in einen sauberen,
analysierbaren Zustand bringen, bevor er mit dem
Produktkatalog (items) zusammengeführt wird.

Der train-Datensatz enthält 2'756'003 Transaktionen mit
11 Spalten: lineID, day, pid, adFlag, availability,
competitorPrice, click, basket, order, price, revenue

In [62]:
## 1. Einlesen der Events (train.csv)
import pandas as pd

train = pd.read_csv('C:/Users/anith/PycharmProjects/DataPreProcessing_final/data/train.csv', header=0, sep='|')

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2756003 entries, 0 to 2756002
Data columns (total 11 columns):
 #   Column           Dtype  
---  ------           -----  
 0   lineID           int64  
 1   day              int64  
 2   pid              int64  
 3   adFlag           int64  
 4   availability     int64  
 5   competitorPrice  float64
 6   click            int64  
 7   basket           int64  
 8   order            int64  
 9   price            float64
 10  revenue          float64
dtypes: float64(3), int64(8)
memory usage: 231.3 MB


## 2.1 Duplikate prüfen

In [63]:
# Duplikate prüfen
print(f"Duplikate in train: {train.duplicated().sum()}")

Duplikate in train: 0


## 2.2 – Datentypen

Prüfung und Anpassung der Datentypen aller Spalten in train.

**Binäre Spalten** (`adFlag`, `click`, `basket`, `order`):
Diese Spalten enthalten ausschliesslich 0/1 Werte und bleiben als `int64` — sie sind direkt vom Modell nutzbar ohne weiteres Encoding.

**`availability`**:
Enthält 4 ordinale Werte (1–4). Eine Analyse der Kaufrate pro Verfügbarkeitsstufe bestätigt eine klare Rangfolge:

| availability | Kaufrate |
|---|---|
| 1 | 26.8% (höchste Verfügbarkeit) |
| 2 | 14.6% |
| 3 | 10.4% |
| 4 | 0.0% (kaum verfügbar) |

Da die Rangfolge inhaltlich bedeutsam ist, wird `availability` als ordinale Kategorie gespeichert (`ordered=True`). Dies macht explizit sichtbar, dass es sich um eine geordnete Gruppe handelt und keine Zahl, mit der gerechnet werden kann.

Alle übrigen Spalten (`int64`, `float64`) behalten ihren ursprünglichen Dtype, da sie numerische Werte repräsentieren mit denen gerechnet wird.

In [64]:
print(train['availability'].value_counts())

availability
1    2515572
2     185194
3      44893
4      10344
Name: count, dtype: int64


In [65]:
# Kaufrate pro availability
availability_check = train.groupby('availability').agg(
    anzahl_events = ('order', 'count'),
    anzahl_orders = ('order', 'sum'),
    kaufrate      = ('order', 'mean')
).reset_index()

print(availability_check)

   availability  anzahl_events  anzahl_orders  kaufrate
0             1        2515572         673456  0.267715
1             2         185194          26980  0.145685
2             3          44893           4651  0.103602
3             4          10344              3  0.000290


In [66]:
from pandas.api.types import CategoricalDtype

# Ordinale Kategorie definieren (1 = beste Verfügbarkeit)
availability_dtype = CategoricalDtype(categories=[1, 2, 3, 4], ordered=True)
train['availability'] = train['availability'].astype(availability_dtype)

# Kontrolle
print(train['availability'].dtype)
print(train['availability'].value_counts().sort_index())

category
availability
1    2515572
2     185194
3      44893
4      10344
Name: count, dtype: int64


### Missing Value

In [67]:
# Fehlende Werte in train prüfen
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
print(pd.DataFrame({'Anzahl': missing, 'Prozent': missing_pct}))

                 Anzahl  Prozent
lineID                0     0.00
day                   0     0.00
pid                   0     0.00
adFlag                0     0.00
availability          0     0.00
competitorPrice  100687     3.65
click                 0     0.00
basket                0     0.00
order                 0     0.00
price                 0     0.00
revenue               0     0.00


Prüfung auf fehlende Werte in allen Spalten von train.

Ergebnis: Nur `competitorPrice` weist fehlende Werte auf
(100'687 Zeilen, 3.65%).

Das Fehlen ist inhaltlich bedeutsam — es signalisiert dass
kein Konkurrenzprodukt für dieses Produkt existiert und ist
somit keine zufällige Lücke sondern eine eigene Information.

**Entscheidung:** Behandlung erst nach dem Merge, da
`competitorPrice` im Verhältnis zu `price` und `rrp`
beurteilt werden muss. Nach dem Merge wird ein binäres
Flag `competitorPrice_missing` erstellt und anschliessend
eine Imputation durchgeführt.

# Merge items und train

Ziel: Den bereinigten Produktkatalog (items) mit den
bereinigten Transaktionsdaten (train) zusammenführen.

Merge-Schlüssel: pid
Merge-Typ: left join → alle Transaktionen in train bleiben
erhalten, auch wenn kein passendes Produkt in items existiert.

In [68]:
# Merge
merged = train.merge(items, on='pid', how='left')

# Kontrolle
print(f"train shape:  {train.shape}")
print(f"items shape:  {items.shape}")
print(f"merged shape: {merged.shape}")

# Prüfen ob alle pids in train auch in items vorhanden sind
missing_pids = train['pid'].isin(items['pid'])
print(f"\npids ohne Match in items: {(~missing_pids).sum()}")

train shape:  (2756003, 11)
items shape:  (22035, 22)
merged shape: (2756003, 32)

pids ohne Match in items: 0


### Outliers

In [69]:
print(merged['pack_size'].describe())
print(f"\nWerte über 10'000: {(merged['pack_size'] > 10000).sum()}")
print(merged[merged['pack_size'] > 10000][['content', 'unit', 'pack_size']].head(10))

count    2.755922e+06
mean     7.908716e+01
std      1.453704e+02
min      2.500000e-01
25%      1.500000e+01
50%      5.000000e+01
75%      1.000000e+02
max      1.000000e+04
Name: pack_size, dtype: float64

Werte über 10'000: 0
Empty DataFrame
Columns: [content, unit, pack_size]
Index: []


In [70]:
print(merged[merged['unit_norm'] == 'ml'][['content', 'unit_norm', 'pack_size']].head(10))
print(merged[merged['unit_norm'] == 'cm'][['content', 'unit_norm', 'pack_size']].head(10))

   content unit_norm  pack_size
0       50        ml       50.0
2     2X50        ml       50.0
5     1000        ml     1000.0
7      100        ml      100.0
8       50        ml       50.0
11     100        ml      100.0
13       3        ml        3.0
17     100        ml      100.0
21      50        ml       50.0
23     500        ml      500.0
       content unit_norm  pack_size
1366        30        cm     3000.0
6714        30        cm     3000.0
6782        50        cm     5000.0
9502        50        cm     5000.0
11209       30        cm     3000.0
14056  100X220        cm      220.0
15520       40        cm     4000.0
15644       40        cm     4000.0
15821  100X220        cm      220.0
17778       30        cm     3000.0


In [71]:
print(items['unit'].value_counts())

unit
ST    10061
ML     7914
G      3878
P       134
L        28
M        13
KG        6
CM        1
Name: count, dtype: int64


In [72]:
print(items[items['unit'] == 'M'][['content', 'unit', 'rrp', 'category_norm']].head(10))

     content unit    rrp category_norm
2855      30    M   2.54           142
5382      20    M   4.39            -1
5969      35    M   7.92           142
6122      20    M   2.19           142
6461      50    M   4.51            -1
7940     1X5    M  18.67            12
9598      50    M   3.29            -1
9638      40    M   4.39           142
9639      50    M   3.62            -1
9640      50    M   3.62            -1


In [73]:
# Prüfen ob Kategorie 142 hauptsächlich ML Produkte hat
print(items[items['category_norm'].astype(str) == '142']['unit'].value_counts())

unit
ST    114
P      11
M       5
Name: count, dtype: int64


In [74]:
# Welche Einheiten haben ähnliche rrp und pack_size wie M?
print("Einheit M:")
print(items[items['unit'] == 'M'][['content', 'unit', 'rrp']].describe())

print("\nEinheit ML:")
print(items[items['unit'] == 'ML'][['content', 'unit', 'rrp']].describe())

print("\nEinheit ST:")
print(items[items['unit'] == 'ST'][['content', 'unit', 'rrp']].describe())

Einheit M:
             rrp
count  13.000000
mean    5.351538
std     4.271937
min     2.190000
25%     3.330000
50%     4.390000
75%     5.280000
max    18.670000

Einheit ML:
               rrp
count  7914.000000
mean     16.490322
std      15.091904
min       0.360000
25%       8.250000
50%      13.200000
75%      19.750000
max     225.500000

Einheit ST:
                rrp
count  10061.000000
mean      20.891346
std       24.168030
min        0.070000
25%        6.680000
50%       13.700000
75%       26.380000
max      404.970000


In [75]:
# Alle relevanten numerischen Spalten auf einmal prüfen
outlier_cols = ['competitorPrice', 'price', 'rrp', 'revenue', 'pack_n']

for col in outlier_cols:
    q1  = merged[col].quantile(0.25)
    q3  = merged[col].quantile(0.75)
    iqr = q3 - q1
    lb  = q1 - 1.5 * iqr
    ub  = q3 + 1.5 * iqr

    n_outlier = ((merged[col] < lb) | (merged[col] > ub)).sum()
    n_zero    = (merged[col] == 0).sum()

    print(f"{'='*50}")
    print(f"Spalte: {col}")
    print(f"  Min:      {merged[col].min():.2f}")
    print(f"  Max:      {merged[col].max():.2f}")
    print(f"  Mean:     {merged[col].mean():.2f}")
    print(f"  Median:   {merged[col].median():.2f}")
    print(f"  IQR-lb:   {lb:.2f}")
    print(f"  IQR-ub:   {ub:.2f}")
    print(f"  Outlier:  {n_outlier:,} ({n_outlier/len(merged)*100:.2f}%)")
    print(f"  Nullwerte:{n_zero:,} ({n_zero/len(merged)*100:.2f}%)")

Spalte: competitorPrice
  Min:      0.00
  Max:      264.59
  Mean:     12.77
  Median:   8.99
  IQR-lb:   -8.89
  IQR-ub:   29.43
  Outlier:  194,482 (7.06%)
  Nullwerte:976 (0.04%)
Spalte: price
  Min:      0.02
  Max:      378.84
  Mean:     13.85
  Median:   9.85
  IQR-lb:   -9.67
  IQR-ub:   32.12
  Outlier:  204,491 (7.42%)
  Nullwerte:0 (0.00%)
Spalte: rrp
  Min:      0.07
  Max:      404.97
  Mean:     18.31
  Median:   13.17
  IQR-lb:   -12.97
  IQR-ub:   42.91
  Outlier:  191,821 (6.96%)
  Nullwerte:0 (0.00%)
Spalte: revenue
  Min:      0.00
  Max:      887.70
  Mean:     3.75
  Median:   0.00
  IQR-lb:   -2.90
  IQR-ub:   4.83
  Outlier:  587,133 (21.30%)
  Nullwerte:2,050,913 (74.42%)
Spalte: pack_n
  Min:      1.00
  Max:      100.00
  Mean:     1.55
  Median:   1.00
  IQR-lb:   1.00
  IQR-ub:   1.00
  Outlier:  116,214 (4.22%)
  Nullwerte:0 (0.00%)


In [76]:
from scipy.stats import mstats

def winsorize_col(df, col, limits=(0.01, 0.01)):
    """
    Winsorisiert eine Spalte und speichert sie als _clean Variante.
    limits: (unteres Limit, oberes Limit) als Anteil (z.B. 0.01 = 1%)
    """
    # Nur nicht-fehlende Werte winsorisieren
    mask = df[col].notna()
    df[f'{col}_clean'] = df[col].copy()
    df.loc[mask, f'{col}_clean'] = mstats.winsorize(
        df.loc[mask, col], limits=limits
    )

    # Kontrolle
    print(f"{'='*50}")
    print(f"Spalte: {col}")
    print(f"  Vorher  – Min: {df[col].min():.2f}, Max: {df[col].max():.2f}, Mean: {df[col].mean():.2f}")
    print(f"  Nachher – Min: {df[f'{col}_clean'].min():.2f}, Max: {df[f'{col}_clean'].max():.2f}, Mean: {df[f'{col}_clean'].mean():.2f}")

# Anwenden
winsorize_col(merged, 'competitorPrice', limits=(0.01, 0.01))
winsorize_col(merged, 'price',           limits=(0.01, 0.01))
winsorize_col(merged, 'rrp',             limits=(0.01, 0.01))

Spalte: competitorPrice
  Vorher  – Min: 0.00, Max: 264.59, Mean: 12.77
  Nachher – Min: 1.19, Max: 68.63, Mean: 12.57
Spalte: price
  Vorher  – Min: 0.02, Max: 378.84, Mean: 13.85
  Nachher – Min: 1.21, Max: 65.95, Mean: 13.59
Spalte: rrp
  Vorher  – Min: 0.07, Max: 404.97, Mean: 18.31
  Nachher – Min: 1.81, Max: 108.66, Mean: 18.10


In [77]:
# Flag erstellen
merged['competitorPrice_missing'] = merged['competitorPrice'].isna().astype(int)

print(f"Fehlende competitorPrice: {merged['competitorPrice'].isna().sum()}")
print(f"competitorPrice_missing Verteilung:\n{merged['competitorPrice_missing'].value_counts()}")

Fehlende competitorPrice: 100687
competitorPrice_missing Verteilung:
competitorPrice_missing
0    2655316
1     100687
Name: count, dtype: int64


In [78]:
global_median = merged['competitorPrice_clean'].median()
merged['competitorPrice_clean'] = merged['competitorPrice_clean'].fillna(global_median)

# Neue Variablen

In [79]:
#day 7
merged['day_7'] = merged['day'] % 7
merged['day_14'] = merged['day'] % 14
merged['day_30'] = merged['day'] % 30

Preise

In [80]:
# price_diff: Preisdifferenz price - competitorPrice
merged['price_diff'] = merged['price_clean'] - merged['competitorPrice_clean']

# price_discount: Rabatt relativ zur UVP (rrp - price) / rrp
merged['price_discount'] = (merged['rrp_clean'] - merged['price_clean']) / merged['rrp_clean']
merged['price_discount'] = merged['price_discount'].clip(lower=-0.5, upper=1.0)

# price_discount_diff: Konkurrenz-Rabatt relativ zur UVP
merged['price_discount_diff'] = (merged['rrp_clean'] - merged['competitorPrice_clean']) / merged['rrp_clean']

# competitorPrice_discount: Rabattdifferenz vs Konkurrenz
merged['competitorPrice_discount'] = merged['price_discount'] - merged['price_discount_diff']

# Binäre Flags
merged['is_lower_price']      = (merged['price_clean'] < merged['competitorPrice_clean']).astype(int)
merged['is_discount']         = (merged['price_clean'] < merged['rrp_clean']).astype(int)
merged['is_greater_discount'] = (merged['price_discount'] > merged['price_discount_diff']).astype(int)

# Preis pro Einheit
merged['rrp_per_unit']             = merged['rrp_clean']             / merged['pack_size'].replace(0, np.nan)
merged['price_per_unit']           = merged['price_clean']           / merged['pack_size'].replace(0, np.nan)
merged['competitorPrice_per_unit'] = merged['competitorPrice_clean'] / merged['pack_size'].replace(0, np.nan)

# Kontrolle
print(merged[['price_diff', 'price_discount', 'price_discount_diff',
              'competitorPrice_discount', 'is_lower_price',
              'is_discount', 'is_greater_discount',
              'rrp_per_unit', 'price_per_unit',
              'competitorPrice_per_unit']].describe())



         price_diff  price_discount  price_discount_diff  \
count  2.756003e+06    2.756003e+06         2.756003e+06   
mean   1.145087e+00    2.332233e-01         2.811464e-01   
std    4.663121e+00    1.488563e-01         2.316193e-01   
min   -5.369000e+01   -5.000000e-01        -1.771486e+01   
25%   -4.000000e-02    9.566044e-02         2.003082e-01   
50%    7.300000e-01    2.418182e-01         2.850730e-01   
75%    1.910000e+00    3.174061e-01         3.746017e-01   
max    6.476000e+01    9.595391e-01         9.890484e-01   

       competitorPrice_discount  is_lower_price   is_discount  \
count              2.756003e+06    2.756003e+06  2.756003e+06   
mean              -4.792310e-02    2.779943e-01  9.830316e-01   
std                2.249367e-01    4.480106e-01  1.291530e-01   
min               -1.082611e+00    0.000000e+00  0.000000e+00   
25%               -1.262580e-01    0.000000e+00  1.000000e+00   
50%               -6.310680e-02    0.000000e+00  1.000000e+00   
75% 

In [81]:
# ============================================================
# AGGREGATIONS-FEATURES
# ============================================================

# pid_total_events: Gesamtanzahl Events pro pid
pid_events = merged.groupby('pid').size().rename('pid_total_events')
merged = merged.merge(pid_events, on='pid', how='left')

# click_time, basket_time, order_time: Anzahl Events pro pid
pid_agg = merged.groupby('pid').agg(
    click_time  = ('click',  'sum'),
    basket_time = ('basket', 'sum'),
    order_time  = ('order',  'sum'),
).reset_index()
merged = merged.merge(pid_agg, on='pid', how='left')

# group12_order: Orders innerhalb der ersten 2 Zeichen von group
merged['group12'] = merged['group'].astype(str).str[:2]
g12 = merged.groupby('group12')['order'].sum().rename('group12_order')
merged = merged.merge(g12, on='group12', how='left')
merged.drop(columns=['group12'], inplace=True)

# week_order: Orders pro Wochentag
week_ord = merged.groupby('day_7')['order'].sum().rename('week_order')
merged = merged.merge(week_ord, on='day_7', how='left')

# ------------------------------------------------------------
# pid_segment: Head (Top 10%), Mid, Tail (Bottom 50%)
# ------------------------------------------------------------
p90 = pid_events.quantile(0.90)
p50 = pid_events.quantile(0.50)

def segment(n):
    if n >= p90:   return 'Head'
    elif n >= p50: return 'Mid'
    else:          return 'Tail'

seg_map = pid_events.apply(segment)
merged['pid_segment'] = merged['pid'].map(seg_map)

# Kontrolle
print(merged[['pid_total_events', 'click_time', 'basket_time',
              'order_time', 'group12_order', 'week_order']].describe())
print(f"\npid_segment:\n{merged['pid_segment'].value_counts()}")

       pid_total_events    click_time   basket_time    order_time  \
count      2.756003e+06  2.756003e+06  2.756003e+06  2.756003e+06   
mean       2.354465e+03  5.926474e+02  1.285541e+03  4.762769e+02   
std        7.575748e+03  8.003400e+02  7.235054e+03  1.032245e+03   
min        1.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
25%        1.570000e+02  9.300000e+01  1.800000e+01  2.600000e+01   
50%        4.950000e+02  2.830000e+02  6.500000e+01  1.030000e+02   
75%        1.581000e+03  8.300000e+02  2.500000e+02  4.280000e+02   
max        5.378500e+04  5.006000e+03  5.240600e+04  8.473000e+03   

       group12_order    week_order  
count   2.756003e+06  2.756003e+06  
mean    6.557295e+04  1.034446e+05  
std     2.804563e+04  1.860938e+04  
min     1.052000e+03  6.367700e+04  
25%     4.029900e+04  9.496400e+04  
50%     7.692400e+04  1.088870e+05  
75%     8.975100e+04  1.184820e+05  
max     9.719700e+04  1.212460e+05  

pid_segment:
pid_segment
Head    1786951
Mid

In [82]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2756003 entries, 0 to 2756002
Data columns (total 56 columns):
 #   Column                    Dtype   
---  ------                    -----   
 0   lineID                    int64   
 1   day                       int64   
 2   pid                       int64   
 3   adFlag                    int64   
 4   availability              category
 5   competitorPrice           float64 
 6   click                     int64   
 7   basket                    int64   
 8   order                     int64   
 9   price                     float64 
 10  revenue                   float64 
 11  manufacturer              int64   
 12  group                     object  
 13  content                   object  
 14  unit                      object  
 15  pharmForm                 object  
 16  genericProduct            int64   
 17  salesIndex                int64   
 18  category                  float64 
 19  campaignIndex             object  
 20  rr

In [83]:
merged.to_csv('merged_clean.csv', index=False)
print(f"Gespeichert: merged_clean.csv")
print(f"Shape: {merged.shape}")

Gespeichert: merged_clean.csv
Shape: (2756003, 56)
